# Rastro Lab: el mismo modelo, explicado con la evidencia de Rastro

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/germanbraun/Rastro/blob/main/lab/notebooks/rastro-lab.ipynb)

**La pregunta:** ¿cuánto gas va a dar un pozo nuevo de shale gas de Vaca Muerta en sus meses 2 a 7?

Se entrena **el mismo modelo** (una regresión lineal) dos veces, con los mismos pozos:

- **A · sin Rastro:** las variables salen de las tablas crudas de la Secretaría de Energía, columna por columna.
- **B · con Rastro:** las mismas variables salen de Rastro, la capa semántica: cada una es un concepto del
  diccionario de negocio, con su fuente, su regla y su evidencia.

Lo que muestra este cuaderno, con datos públicos reales:

1. **La precisión no cambia:** A y B predicen igual de bien. Rastro no hace magia con el modelo.
2. **Lo que cambia es la explicación:** con Rastro, cada predicción dice en qué se apoya, de dónde sale cada dato
   y cuáles hay que mirar dos veces.
3. **Rastro sabe qué predicciones revisar:** las que usan una unión revisada por IA o un dato que llegó tarde.
4. **Una corrección llega al modelo:** un dato corregido una vez en Rastro cambia lo que entra al modelo.
5. Al final, otros algoritmos probados, y por qué el modelo publicado es lineal.

Corre sin Postgres ni Rastro: lee una copia de lo que lee el modelo (`lab/notebooks/datos/`, guardada con
`lab/gas/snapshot.py`) y usa **el mismo código que la app** (`lab/gas/modelo.py`). Al final comprueba que sus
números son los de la corrida publicada.

In [1]:
# Find the repository (or clone it, in Colab) and import the same model code the app uses
import os
import subprocess
import sys
from pathlib import Path

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lab/gas/modelo.py").exists()), None)
if root is None:
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/germanbraun/Rastro.git"], check=True)
    root = Path("Rastro").resolve()
os.chdir(root)
sys.path.insert(0, str(root / "lab" / "gas"))

import hashlib
import json

import pandas as pd

import modelo  # noqa: E402

pd.set_option("display.max_colwidth", 200)
print("Código del modelo:", modelo.__file__.replace(str(root) + os.sep, ""), "· versión", modelo.version())

Código del modelo: lab/gas/modelo.py · versión 1b8209a1


## 0 · Los datos, y que sean los de la corrida publicada

Cada archivo tiene su huella (SHA-256) en `manifest.json`, junto con la versión del modelo que los leyó.

In [2]:
DATA = root / "lab" / "notebooks" / "datos"
manifest = json.loads((DATA / "manifest.json").read_text())
for name, spec in manifest["files"].items():
    assert hashlib.sha256((DATA / name).read_bytes()).hexdigest() == spec["sha256"], f"{name} changed"
assert manifest["model_version"] == modelo.version(), "modelo.py changed after this snapshot: save a new one"

gas = pd.read_csv(DATA / "gas.csv", dtype={"idpozo": str})
raw = pd.read_csv(DATA / "a.csv", dtype={"idpozo": str})          # A: the raw tables' columns
rastro = pd.read_csv(DATA / "b.csv.gz")                             # B: Rastro's concepts, with rule and evidence
socios = pd.read_csv(DATA / "socios.csv")
cambios = pd.read_csv(DATA / "cambios.csv")
publicadas = pd.read_csv(DATA / "predicciones.csv", dtype={"idpozo": str})

A, B = modelo.preparar(gas, raw, rastro)
ctx = modelo.contexto(socios, cambios)
r = modelo.modelos(A, B)
print(f"Corrida publicada: {manifest['run']} · guardada el {manifest['saved_utc'][:10]}")
print(f"Pozos: {len(B)} ({(B.inicio < modelo.PRUEBA).sum()} para entrenar, {(B.inicio >= modelo.PRUEBA).sum()} de 2025 para probar)")

Corrida publicada: 20261026-212547-1b8209a1 · guardada el 2026-10-28


Pozos: 559 (510 para entrenar, 49 de 2025 para probar)


## 1 · El mismo modelo, la misma precisión

El error medio en los pozos de 2025 (que el modelo no vio), en millones de m³. Como referencia, sin modelo:
la mediana de todos los pozos, o la de su operadora o su concesión.

In [3]:
ref = modelo.referencias(A)
pd.DataFrame([
    {"Cómo se predice": "Sin modelo: la mediana de todos los pozos", "Error medio": ref["todos"]},
    {"Cómo se predice": "Sin modelo: la mediana de su concesión", "Error medio": ref["concesion"]},
    {"Cómo se predice": "A · el modelo, con las tablas crudas", "Error medio": r["A"]["error"], "R²": r["A"]["r2"]},
    {"Cómo se predice": "B · el modelo, con Rastro", "Error medio": r["B"]["error"], "R²": r["B"]["r2"]},
]).round(2)

,Cómo se predice,Error medio,R²
0,Sin modelo: la mediana de todos los pozos,22.35,NaN
1,Sin modelo: la mediana de su concesión,17.80,NaN
2,"A · el modelo, con las tablas crudas",16.65,0.46
3,"B · el modelo, con Rastro",16.67,0.46


**A y B se equivocan casi lo mismo.** Rastro no promete un modelo más preciso: promete saber en qué se apoya
cada predicción. Eso es lo que sigue.

## 1b · El algoritmo: una regresión lineal con regularización (ridge)

El modelo es deliberadamente simple, para que cada predicción se pueda explicar **exacta**:

- **Entradas:** la operadora y la concesión como categorías (una columna 0/1 por empresa y por concesión) y, como
  números, la participación, la fractura (rama horizontal, etapas, arena, agua y presión) y la profundidad.
  Un valor que falta se completa con el promedio del entrenamiento: así no suma ni resta.
- **Ajuste (ridge):** con las columnas centradas en su promedio de entrenamiento $ar{x}$ y el gas centrado en
  su promedio $ar{y}$, los coeficientes salen de
  $$eta = (X_c^	op X_c + lpha I)^{-1} X_c^	op (y - ar{y}), \qquad lpha = 1.$$
- **Predicción:** $\hat{y} = ar{y} + \sum_j (x_j - ar{x}_j)\,eta_j$. Cada término es el **aporte** de una
  variable, en millones de m³: la explicación no es una aproximación, es la cuenta misma.

Este es el código, leído de `lab/gas/modelo.py` (el mismo que usa la app):

In [4]:
import inspect

print(inspect.getsource(modelo.ajustar))
print(inspect.getsource(modelo.entrenar))

def ajustar(df, categorias, numeros, prueba=PRUEBA, alpha=1.0):
    """El ajuste de la regresión ridge: la tabla de entrada (categorías en columnas 0/1, faltantes con el promedio
    del entrenamiento), sus promedios en el entrenamiento, el gas promedio y un coeficiente por columna."""
    entreno = (df.inicio < prueba).values
    x = pd.get_dummies(df[categorias].astype(str), prefix_sep="=", dtype=float)
    n = df[numeros].astype(float)
    n = n.fillna(n[entreno].mean())         # lo que falta, con el promedio del entrenamiento: no suma ni resta
    x = pd.concat([x, n], axis=1)
    media_x, media_y = x[entreno].mean(), df.gas[entreno].mean()
    xc = (x[entreno] - media_x).values
    coef = np.linalg.solve(xc.T @ xc + alpha * np.eye(xc.shape[1]), xc.T @ (df.gas[entreno].values - media_y))
    return x, n, entreno, media_x, media_y, coef

def entrenar(df, categorias, numeros, prueba=PRUEBA, alpha=1.0):
    """Regresión lineal con un poco de regularización (ridge). Devuelve la predic

**Comprobaciones:** el mismo ajuste con la regresión ridge de scikit-learn da las mismas predicciones, y la
predicción de un pozo es exactamente el pozo promedio más la suma de sus aportes.

In [5]:
from sklearn.linear_model import Ridge

x, _, entreno, _, _, _ = modelo.ajustar(B, **modelo.VARIABLES["B"])
sk = Ridge(alpha=1.0).fit(x[entreno].values, B.gas[entreno].values)
print(f"Columnas del modelo: {x.shape[1]} · pozos de entrenamiento: {entreno.sum()}")
print(f"Diferencia máxima con scikit-learn: {abs(sk.predict(x.values) - r['B']['pred']).max():.1e} millones de m³")

i = B.index[B.idpozo == modelo.pozo_ejemplo(B)][0]
suma = r["B"]["base"] + r["B"]["aportes"].loc[i].sum()
print(f"Pozo {B.idpozo[i]}: predicción {r['B']['pred'][i]:.4f} = pozo promedio {r['B']['base']:.4f} "
      f"+ suma de aportes {r['B']['aportes'].loc[i].sum():.4f} = {suma:.4f}")
assert abs(suma - r["B"]["pred"][i]) < 1e-9

Columnas del modelo: 36 · pozos de entrenamiento: 510
Diferencia máxima con scikit-learn: 9.0e-13 millones de m³
Pozo 165404: predicción 54.1477 = pozo promedio 51.6357 + suma de aportes 2.5120 = 54.1477


## 2 · Una predicción, explicada sin Rastro y con Rastro

El pozo de ejemplo es el que abre la pestaña Rastro Lab de la app.

In [6]:
pozo = modelo.pozo_ejemplo(B)
sin = modelo.explicacion_a(r, A, pozo)
print(f"Pozo {pozo} · predicción sin Rastro: {sin['pred']:.1f} millones de m³")
pd.DataFrame(sin["filas"]).sort_values("aporte", key=abs, ascending=False).round(2).rename(
    columns={"columna": "Columna de la tabla", "valor": "Valor", "aporte": "Aporte (millones de m³)"})

Pozo 165404 · predicción sin Rastro: 54.1 millones de m³


,Columna de la tabla,Valor,Aporte (millones de m³)
4,cantidad_fracturas,53,9.25
1,area,LCA,-7.12
3,longitud_rama_horizontal_m,3186,-4.55
6,agua_inyectada_m3,106919,2.35
2,porcentaje,50,1.96
0,empresa,PLUSPETROL S.A.,1.63
8,profundidad_vertical_mt,2895.06,-0.72
5,arena_tn,12471.8,-0.40
7,presion_maxima_psi,11472,0.07


**Sin Rastro**, la explicación es lo que da cualquier herramienta estándar: una lista de columnas con su aporte.
Dice cuánto pesa `porcentaje` o `empresa`, pero no de dónde salió cada valor, si alguien lo corrigió, si la
empresa es la misma en las dos tablas o si el dato llegó tarde.

In [7]:
con = modelo.explicacion_b(r, B, ctx, pozo)
print(con["resumen"])
pd.DataFrame(con["factores"])[["factor", "aporte", "por_que"]].round(2).rename(
    columns={"factor": "Factor", "aporte": "Aporte (millones de m³)", "por_que": "Por qué"})

Se esperan 54,1 millones de m³: un pozo promedio da 51,6; estar en La Calera con Pluspetrol resta 3,6, cómo se fracturó suma 6,8 y su profundidad resta 0,7.


,Factor,Aporte (millones de m³),Por qué
0,"La Calera, con Pluspetrol",-3.57,"Los 73 pozos de La Calera que vio el modelo (todos de Pluspetrol) dieron una mediana de 45,5 millones de m³, menos que la de todos los pozos (52,2)."
1,Cómo se fracturó,6.80,"Pesan sobre todo: etapas de fractura 53 (típico: 39), +9,3; largo de la rama horizontal 3.186 m (típico: 2.278), −4,5. Un pozo con más rama tiene más etapas: el modelo reparte entre las dos un mis..."
2,Su profundidad,-0.72,"Está a 2.895 m de profundidad vertical (típico: 2.788), de su trayectoria; en el entrenamiento, los pozos más someros dieron más gas."


**Con Rastro**, la misma cuenta se lee en conceptos de negocio: la concesión con su operadora y sus socios, cómo
se fracturó el pozo y su profundidad. Y cada concepto trae **los hechos que lo respaldan**:

In [8]:
pd.DataFrame(con["hechos"])[["hecho", "valor", "marca", "regla", "evidencia", "fuentes"]].rename(columns={
    "hecho": "Hecho", "valor": "Valor", "marca": "A mirar", "regla": "Regla", "evidencia": "Evidencia", "fuentes": "Fuente"})

,Hecho,Valor,A mirar,Regla,Evidencia,Fuente
0,Operadora,Pluspetrol,"unión revisada por IA, a confirmar por una persona",rige quien declara el pozo en su primer mes,"Primer mes con producción: 02/2025. Lo declara PLUSPETROL S.A como pozo gasífero, en LCA.",Capítulo IV de la Secretaría de Energía (declaraciones juradas)
1,Socios,"Pluspetrol 50 %, YPF S.A. 50 %",,texto de socios del mapa,PLUSPETROL S.A.: 50 %,Mapa de concesiones de la Secretaría de Energía (texto de socios)
2,Fractura,"3.186 m de rama, 53 etapas","unión revisada por IA, a confirmar por una persona · cargada 2 meses después de que arrancó el pozo","las fracturas informadas hasta el mes en que arrancó el pozo, sumadas","Fractura informada por PLUSPETROL S.A desde 04/10/2024: rama horizontal de 3186 m, 53 etapas, 12472 t de arena, 106919 m³ de agua, presión máxima 11472 psi. Cargada en el portal el 24/04/2025.","Datos de fractura de pozos (Adjunto IV), Secretaría de Energía"
3,Trayectoria,2.895 m de profundidad vertical,,una por pozo: la de rama más larga entre las perforadas antes de que arrancara,"Trayectoria PLU.Nq.LCa-3291(h): profundidad vertical de 2895 m, rama horizontal de 3209 m, perforado hasta el 27/06/2024.","Trayectorias de pozo de Vaca Muerta, Secretaría de Energía"


Lo que la explicación con Rastro dice y la de las columnas no puede decir:

- **De qué fuente sale cada dato, con qué regla y con qué evidencia** (por ejemplo, cómo se unió el nombre de la
  empresa en las declaraciones juradas con el de los datos de fractura).
- **Qué hay que mirar dos veces** (columna «A mirar»): una unión de nombres que revisó un asistente de IA y espera
  la confirmación de una persona, un dato corregido con un documento, o una fractura que se cargó en el portal
  **después** de que el pozo arrancó (el modelo no la habría tenido ese día).

## 3 · Rastro sabe qué predicciones revisar

Como cada predicción dice qué datos usó, se puede preguntar cuáles se apoyan en algo dudoso, en todos los pozos
de 2025 a la vez.

In [9]:
dudosas, probadas = modelo.dudosos(B)
tarde = modelo.al_arrancar(r, B)
prueba = B[B.inicio >= modelo.PRUEBA]
marcas = pd.concat([prueba.reglaNombre.map(modelo.marca), prueba.reglaSocio.map(modelo.marca)])
pd.DataFrame([
    {"Qué": "Pozos de 2025 predichos", "Cuántos": probadas},
    {"Qué": "…que se apoyan en una unión o un dato que no sale tal cual de la fuente", "Cuántos": dudosas},
    {"Qué": "…con datos de fractura", "Cuántos": tarde["con_fractura"]},
    {"Qué": "…cuya fractura se cargó después de que arrancó el pozo", "Cuántos": tarde["tarde"]},
    {"Qué": "Cambio medio de su predicción si se predice el día que arrancó (millones de m³)", "Cuántos": round(tarde["cambio_medio"], 1)},
])

,Qué,Cuántos
0,Pozos de 2025 predichos,49.0
1,…que se apoyan en una unión o un dato que no sale tal cual de la fuente,24.0
2,…con datos de fractura,43.0
3,…cuya fractura se cargó después de que arrancó el pozo,36.0
4,Cambio medio de su predicción si se predice el día que arrancó (millones de m³),5.1


Sin Rastro, estas preguntas no tienen respuesta: las columnas no guardan de dónde vino cada valor ni cuándo llegó.

## 4 · Una corrección, hecha una vez, llega al modelo

En Aguada Pichana Este, el mapa de concesiones dice que Total tiene el 27,27 %. Un documento (el Acuerdo Marco de
Neuquén) dice 55 %. En Rastro se corrigió **una vez**, con el documento, y el modelo con Rastro ya lo usa; el
modelo sin Rastro sigue leyendo el texto del mapa.

In [10]:
corregidos = B.reglaSocio.map(modelo.marca) == modelo.texto("marca_documento")
ape = B[corregidos].merge(A[["idpozo", "porcentaje"]], on="idpozo")
print(f"Pozos con la participación corregida: {corregidos.sum()} · con Rastro: {ape.pct.iloc[0]:g} % · "
      f"sin Rastro (el mapa): {ape.porcentaje.iloc[0]:g} %")
print("Evidencia de la corrección:", B[corregidos].textoSocio.iloc[0][:300])

# What the model with Rastro would predict without the correction: the same B with the map's percentage
B_mapa = B.copy()
B_mapa.loc[corregidos, "pct"] = B_mapa.loc[corregidos, "idpozo"].map(A.set_index("idpozo").porcentaje)
sin_correccion = modelo.entrenar(B_mapa, **modelo.VARIABLES["B"])
cambio = pd.Series(r["B"]["pred"] - sin_correccion["pred"], index=B.idpozo)
de_prueba = (B.inicio >= modelo.PRUEBA).values
print(f"Predicciones de 2025 que cambian por la corrección: {(cambio[de_prueba].abs() > 0.005).sum()} de {de_prueba.sum()}; "
      f"el cambio más grande: {cambio[de_prueba].abs().max():.2f} millones de m³")

Pozos con la participación corregida: 74 · con Rastro: 55 % · sin Rastro (el mapa): 27.27 %
Evidencia de la corrección: Corrección revisada: TOTAL AUSTRAL S.A.: 55.0 %. El mapa de la Secretaría de Energía reparte la concesión en partes iguales (27,27 % para Total, Wintershall e YPF; 18,18 % para Pan American); el Acuerdo Marco lista 55 %, 22,5 %, 16,9 % y 5,6 %. Detectado en la verificación humana. Documento: Acuerdo
Predicciones de 2025 que cambian por la corrección: 37 de 49; el cambio más grande: 1.82 millones de m³


La corrección cambia lo que **entra** al modelo para cada pozo de esa concesión, y su explicación lo marca
(«dato corregido con documento»). Sin Rastro, el mismo arreglo habría que hacerlo en cada consulta y en el código
del modelo. En la app, el momento 4 de la demo mide lo mismo sobre la respuesta, el modelo y la explicación.

## 5 · Experimentar con otros algoritmos

Todos los algoritmos que se probaron en Rastro Lab, con las mismas variables y el mismo corte por fecha, para
correrlos y compararlos acá. Cambiá la configuración de la celda siguiente y volvé a correr las dos celdas.

| Nombre | Algoritmo | De dónde sale |
|---|---|---|
| `mediana_concesion` | Sin modelo: la mediana del gas de los pozos de su concesión | referencia |
| `ridge` | **Regresión lineal ridge, el modelo publicado** (sección 1b) | `lab/gas/modelo.py` |
| `arbol` | Un árbol de decisión chico (2 niveles, 10 pozos por hoja) | `lab/gas/compare_trees.py` |
| `bosque` | Bosque aleatorio, 500 árboles, 4 pozos por hoja | `lab/gas/compare_trees.py` |
| `rf`, `xgb`, `lgbm` | Bosque aleatorio, XGBoost y LightGBM con la configuración de la tesis, sobre el logaritmo del gas | `lab/gas/alternative.py` |
| `rf_direct`, `xgb_direct`, `lgbm_direct` | Los mismos tres, sobre el gas directo | `lab/gas/alternative.py` |

Solo `ridge` se descompone **exacto** en aportes, y por eso es el que se explica con la evidencia de Rastro; los
árboles y los ensambles predicen, pero su explicación sería otra aproximación. El *pipeline* completo de la tesis
(`lab/gas/thesis_pipeline.py`) no está acá: necesita sus propias descargas; sus resultados están en
`docs/experiments/gas-thesis/`.

In [11]:
# The experiment's settings: change them and run this cell and the next one
VARIABLES = ["A", "B"]          # "A": the raw tables' columns; "B": Rastro's concepts
CORTE = "2025-01-01"            # train with the wells that started before; test with the ones that started after
OCULTAR_FRACTURA_TARDIA = False # True: each well's fracturing as the source had it the month it started
ALPHA = 1.0                     # ridge's regularization (the published model uses 1.0)
ALGORITMOS = ["mediana_concesion", "ridge", "arbol", "bosque", "rf", "xgb", "lgbm",
              "rf_direct", "xgb_direct", "lgbm_direct"]

In [12]:
import time

import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

import alternative
import compare_trees

corte = pd.Timestamp(CORTE)


def entrada(path):
    """The wells, with the features of path (A or B), as the model would see them."""
    frame = (A if path == "A" else B).copy()
    if OCULTAR_FRACTURA_TARDIA:
        frame, _ = compare_trees.available_inputs(frame, B.fracCargada,
                                                  modelo.FRACTURA_A if path == "A" else modelo.FRACTURA_B)
    return frame


def predecir(nombre, frame, path):
    """One algorithm's predictions for every well, trained only with the wells before the cutoff."""
    variables = modelo.VARIABLES[path]
    train = (frame.inicio < corte).values
    if nombre == "mediana_concesion":
        medianas = frame[train].groupby("area").gas.median()
        return frame.area.map(medianas).fillna(frame.gas[train].median()).to_numpy()
    if nombre == "ridge":
        return modelo.entrenar(frame, prueba=corte, alpha=ALPHA, **variables)["pred"]
    if nombre in ("arbol", "bosque"):
        x = compare_trees.design(frame, variables["categorias"], variables["numeros"], train)
        estimator = (DecisionTreeRegressor(max_depth=2, min_samples_leaf=10, random_state=42) if nombre == "arbol"
                     else RandomForestRegressor(n_estimators=500, min_samples_leaf=4, random_state=42, n_jobs=-1))
        return estimator.fit(x[train], frame.gas[train]).predict(x)
    return alternative.fit(frame, path, nombre, corte)[2]


filas = []
for path in VARIABLES:
    frame = entrada(path)
    prueba = (frame.inicio >= corte).values
    for nombre in ALGORITMOS:
        inicio = time.time()
        try:
            pred = predecir(nombre, frame, path)
        except ImportError as falta:   # XGBoost and LightGBM come with Colab; elsewhere: pip install xgboost lightgbm
            filas.append({"Variables": path, "Algoritmo": nombre, "Nota": f"falta instalar: {falta.name}"})
            continue
        y, p = frame.gas[prueba].to_numpy(), pred[prueba]
        filas.append({"Variables": path, "Algoritmo": nombre, "Error medio": float(np.abs(p - y).mean()),
                      "R²": modelo.r2(p, y), "Segundos": round(time.time() - inicio, 1)})

resultados = pd.DataFrame(filas)
print(f"Entrenamiento: pozos que arrancaron antes de {CORTE} · prueba: {prueba.sum()} pozos"
      + (" · fractura como estaba al arrancar" if OCULTAR_FRACTURA_TARDIA else ""))
resultados.round(2).sort_values(["Variables", "Error medio"]).reset_index(drop=True)

Entrenamiento: pozos que arrancaron antes de 2025-01-01 · prueba: 49 pozos


,Variables,Algoritmo,Error medio,R²,Segundos
0,A,xgb_direct,16.21,0.38,0.1
1,A,rf_direct,16.45,0.43,0.4
2,A,ridge,16.65,0.46,0.0
3,A,lgbm_direct,16.87,0.40,0.1
4,A,bosque,16.92,0.40,0.4
5,A,mediana_concesion,17.80,0.42,0.0
6,A,rf,18.59,0.21,0.5
7,A,xgb,19.09,0.17,0.1
8,A,lgbm,20.12,0.11,0.1
9,A,arbol,21.17,0.12,0.0


Para leer la tabla: el error medio está en millones de m³ de gas de los meses 2 a 7, en pozos que el modelo no
vio, y el R² dice cuánto de la variación entre pozos explica cada modelo. Hay que leerla con cuidado: son pocos
pozos de prueba, y diferencias de medio millón de m³ cambian con la semilla y con el año de corte (con tres años,
en `docs/experiments/gas-trees/`). Que un ensamble se equivoque un poco menos en una configuración no lo hace el
modelo a publicar: **la regresión lineal se publica porque su predicción se descompone exacta en aportes**, y cada
aporte se puede respaldar con la evidencia de Rastro. Y en un mismo algoritmo, A y B dan casi lo mismo: Rastro no
cambia cuánto acierta un modelo, cambia lo que se puede decir de cada predicción (secciones 2 a 4).

## 6 · Comprobación: son los números de la corrida publicada

El cuaderno no copia números: los calcula. Esta celda falla si no coinciden con las predicciones que Rastro
publicó (`lab_gas.prediccion`, la corrida vigente).

In [13]:
mias = pd.DataFrame({"idpozo": B.idpozo.values, "mia": r["B"]["pred"]})
juntas = publicadas.merge(mias, on="idpozo")
diferencia = float((juntas.gas - juntas.mia).abs().max())
assert len(juntas) == len(publicadas), "a published well is missing"
assert diferencia < 0.001, f"the predictions differ from the published run by {diferencia}"
print(f"{len(juntas)} predicciones iguales a las publicadas (diferencia máxima {diferencia:.4f} millones de m³, el redondeo de la base).")

559 predicciones iguales a las publicadas (diferencia máxima 0.0005 millones de m³, el redondeo de la base).


---
**Para ver lo mismo en vivo:** la app de Rastro, pestaña **Rastro Lab** (y el momento 3 de la demo), con cada
dato abierto hasta su registro de origen. Este cuaderno usa datos públicos de la Secretaría de Energía de Argentina,
leídos a través de Rastro; las uniones revisadas por IA esperan todavía la confirmación de una persona.